# Notebook 48R1 - Hybrid Frontier Test
Creates Hybrid25/50/75, Median, Minimum and Maximum ensembles from R1_biharmonic and R3_learned_unet outputs and prepares scoring using Notebook 46 helpers.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import cv2, numpy as np, pandas as pd
from pathlib import Path
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt


In [ ]:
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
RUN=max([p for p in (ROOT/'observed_print_classifier_v8_6_notebook46').iterdir() if p.is_dir()], key=lambda p:p.stat().st_mtime)
TABLES=RUN/'tables'
OUT=RUN/'notebook48R1_hybrid_frontier'
HYBRIDS=OUT/'hybrids'
FIGS=OUT/'figures'
for p in [OUT,HYBRIDS,FIGS]: p.mkdir(parents=True,exist_ok=True)
inventory=pd.read_csv(TABLES/'whole_page_candidate_inventory.csv')
print(RUN)


In [ ]:
def ink(im):
    return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):
    return cv2.Canny(im,70,170)>0
def f1_from_features(ref,pred,reg,tol=1):
    ref=ref&reg; pred=pred&reg
    if not ref.any(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
    q=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
    return float(p),float(q),float(2*p*q/max(p+q,1e-12))


In [ ]:
pairs=inventory[inventory.restorer.isin(['R1_biharmonic','R3_learned_unet'])]
pairs=(pairs.pivot_table(index=['page','source_group','arm'],columns='restorer',values='candidate',aggfunc='first').reset_index())
rows=[]
for _,r in pairs.iterrows():
    b=cv2.imread(str(r['R1_biharmonic']),0)
    u=cv2.imread(str(r['R3_learned_unet']),0)
    if b is None or u is None: continue
    products={
      'Hybrid25':0.75*b+0.25*u,
      'Hybrid50':0.5*b+0.5*u,
      'Hybrid75':0.25*b+0.75*u,
      'Median':np.median(np.stack([b,u]),axis=0),
      'Minimum':np.minimum(b,u),
      'Maximum':np.maximum(b,u)}
    for name,img in products.items():
      img=np.clip(img,0,255).astype(np.uint8)
      fp=HYBRIDS/f'{r.page}_{r.arm}_{name}.png'
      cv2.imwrite(str(fp),img)
      rows.append({'page':r.page,'source_group':r.source_group,'arm':r.arm,'restorer':name,'candidate':str(fp)})
hybrid_inventory=pd.DataFrame(rows)
hybrid_inventory.to_csv(OUT/'hybrid_inventory.csv',index=False)
print('Generated',len(hybrid_inventory),'hybrids')


Copy the page_clean, page_obs and page_auth setup cells from Notebook 46 immediately above the scoring stage, then score hybrid_inventory using the same whole-page scoring block, replacing whole_candidates with hybrid_inventory.
